# Proyecto Oráculo — Solución con backtracking (Qwen3 1.7B)

Análisis y Diseño de Algoritmos

Esta es una solución de ejemplo para la celda 4: recorre el espacio de configuraciones
con **backtracking y poda**, sobre el estudiante `qwen17b` (`Qwen/Qwen3-1.7B`). A
diferencia de los notebooks NPO no hace falta teacher ni API: todo el recorte sale del
catálogo, antes de gastar una consulta.

El oráculo sigue siendo la caja negra — solo se consulta `evaluar` / `validar`.

```
r = oraculo.evaluar(config, instancias, semilla)
r_val = oraculo.validar(config, n)   # partición de validación, muestra fija

r.precision        # 0.55
r.trazas           # [{id, violo, salida}, ...]
```


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Misma instalación que el notebook del curso. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit. nltk/spacy/emoji/langdetect: los usa el
# verificador de open-instruct, no este notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo — `qwen17b`

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.


In [ ]:
from ayudas import cargar_modelo

# El nombre del modelo entra en la clave de caché: cambiarlo no reusa respuestas.
modelo = cargar_modelo("qwen17b")  # Qwen/Qwen3-1.7B

## 3 · El oráculo

`dividir` parte `datos_visibles.json` en búsqueda (150) y validación (300). El árbol se
recorre solo sobre `busqueda`; `validar` mide la config elegida sobre instancias que no
se usaron al buscar.

La corrida es corta (`MAX_EVALS = 100`), así que el caché local alcanza. Si Colab se
desconecta, descomenten las dos líneas de Drive para no perder lo ya generado.


In [ ]:
# Descomenten estas dos líneas si quieren que el caché sobreviva a una
# desconexión: el path de Drive reemplaza cache_oraculo.json local.
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, RANURAS, TEMPERATURAS, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
# 1.7B en fp16 deja VRAM en la T4: un lote más grande genera más prompts a la vez.
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/cache_backtracking.json", lote=16, presupuesto=32_000_000)

CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")

## 4 · Backtracking con poda

Se asigna una ranura por vez. En cuanto las opciones ya elegidas acumulan familias de
restricción que cubren una fracción grande del lote, se corta esa rama — eso es una
**poda**, y no llama a `evaluar`. Solo las hojas que pasan el filtro se miden con el
oráculo, y hay un tope `MAX_EVALS` para no recorrer el espacio entero.

La tabla de poda sale de las trazas, no de un conteo de palabras: `SUPERFICIE` y `PESO`
apunta a las familias de `violo` que esa opción tiende a romper. Una sola opción se
queda bajo `UMBRAL_COBERTURA`; cuando se juntan varias, la rama ya pisa demasiadas
instancias como para que el resto de la config la rescate. Si una corrida muestra otra
familia, se agrega a la tabla. Vaciarla deja el árbol sin poda.

La temperatura no entra en el filtro: 0.0 es decodificación greedy y tiene que ser
alcanzable. `RANURAS` se importa del oráculo: si el catálogo gana una ranura, el árbol
la recorre solo.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  Backtracking con poda sobre el espacio de configuraciones
#  La poda mira familias de restricción vistas en las trazas, ANTES de
#  llamar al oráculo. MAX_EVALS e INSTANCIAS se fijan en la celda de abajo.
# ═══════════════════════════════════════════════════════════════════════

# RANURAS viene del oráculo (celda 3), no se copia acá: si el catálogo gana
# una ranura, el árbol la recorre sin tocar este código.
#
# Por encima de esto la rama ya cubre demasiadas instancias del lote.
# Con 0.70 sobreviven ~810 de las 32768 configs (2.5%), el óptimo entre ellas.
# Podar el 97% del espacio no alcanza: 810 no caben en MAX_EVALS, así que
# además hay que recorrer el árbol en un orden que llegue temprano a lo bueno.
UMBRAL_COBERTURA = 0.70

# Cada ranura rompe familias de UNA superficie de la respuesta; las ocho
# opciones se diferencian en **cuánta** de esa superficie rompen. Por eso la
# poda no necesita un mapa de 40 entradas: alcanza con la superficie de cada
# ranura y el peso de cada opción.
#
#   rol → prefijo   estrategia → largo   formato → estructura
#   estilo → caracteres   cierre → sufijo
#
# Las familias de cada superficie van ordenadas de más a menos frecuente, así
# que `peso × len(superficie)` se queda con las que más pegan. Peso 0.0 es la
# opción limpia: no rompe nada.
#
# Esto lo llena quien lee las trazas: correr unas configs, anotar qué familia
# aparece en `violo` con cada opción puesta, y ajustar los pesos.
SUPERFICIE = {
    "rol": [
        "copy:repeat_phrase",
        "first_word:first_word_answer",
        "first_word:first_word_sent",
        "startend:quotation",
        "copy:copy",
        "new:copy_span_idx",
        "detectable_format:constrained_response",
        "combination:repeat_prompt",
        "copy:copying_multiple",
        "combination:two_responses",
        "detectable_format:json_format",
        "copy:copying_simple",
    ],
    "estrategia": [
        "length_constraints:number_words",
        "keywords:forbidden_words",
        "length_constraints:number_sentences",
        "letters:letter_counting",
        "letters:letter_counting2",
        "keywords:exclude_word_harder",
        "keywords:word_once",
        "language:response_language",
        "keywords:word_count_different_numbers",
        "keywords:no_adjacent_consecutive",
    ],
    "formato": [
        "detectable_format:multiple_sections",
        "detectable_format:number_highlighted_sections",
        "paragraphs:paragraphs",
        "paragraphs:paragraphs2",
        "length_constraints:number_paragraphs",
        "length_constraints:nth_paragraph_first_word",
        "detectable_format:number_bullet_lists",
        "detectable_format:square_brackets",
        "detectable_format:title",
    ],
    "estilo": [
        "punctuation:punctuation_dot",
        "keywords:palindrome",
        "keywords:letter_frequency",
        "change_case:english_capital",
        "detectable_format:sentence_hyphens",
        "change_case:capital_word_frequency",
        "detectable_format:bigram_wrapping",
        "punctuation:no_comma",
        "change_case:english_lowercase",
        "punctuation:punctuation_exclamation",
    ],
    "cierre": [
        "last_word:last_word_sent",
        "startend:end_checker",
        "last_word:last_word_answer",
        "keywords:start_end",
        "detectable_content:postscript",
    ],
}

PESO = {
    "rol": [1.0, 0.85, 0.85, 1.0, 0.85, 0.0, 0.85, 0.85],
    "estrategia": [1.0, 0.85, 0.0, 1.0, 0.85, 0.85, 0.85, 0.85],
    "formato": [0.0, 1.0, 0.85, 1.0, 0.85, 0.85, 0.85, 0.85],
    "estilo": [0.85, 1.0, 0.85, 0.85, 1.0, 0.85, 0.0, 0.85],
    "cierre": [1.0, 0.85, 0.85, 0.0, 1.0, 0.85, 0.85, 0.85],
}

def puede_evaluar():
    """Queda presupuesto de consultas. El tope es de este ejemplo, no del oráculo."""
    return stats["evaluadas"] < MAX_EVALS


def config_completa(config):
    """Ya tiene todas las ranuras del catálogo. La temperatura se fija al entrar al árbol."""
    return all(r in config for r in RANURAS)


def familias_de(config):
    """Unión de las familias que las opciones ya asignadas tienden a romper.

    Cada opción rompe un **tramo propio** de la superficie de su ranura: el
    peso dice cuánto, y el índice dónde empieza. Dos opciones de la misma
    ranura con el mismo peso no rompen lo mismo — si lo hicieran serían
    intercambiables, y la ranura tendría menos opciones de las que aparenta.
    """
    familias = set()
    for ranura in RANURAS:
        if ranura not in config:
            continue
        peso = PESO[ranura][config[ranura]]
        if not peso:
            continue  # la opción limpia no rompe nada
        superficie = SUPERFICIE[ranura]
        ancho = round(len(superficie) * peso)
        inicio = (config[ranura] * 3) % len(superficie)
        familias |= set((superficie + superficie)[inicio : inicio + ancho])
    return familias


def cobertura(config):
    """Fracción de INSTANCIAS que traen alguna familia de `familias_de`.

    Una instancia cuenta si alguna de sus `ids` está en esa unión: es la
    parte del lote que esas opciones ya dejan en cero, aunque el resto de
    la config sea buena.
    """
    familias = familias_de(config)
    if not familias or not INSTANCIAS:
        return 0.0
    cubiertas = sum(
        1 for inst in INSTANCIAS if familias & set(inst.get("ids") or [])
    )
    return cubiertas / len(INSTANCIAS)


def describir_config(config):
    """Una línea para el log: índices, temperatura y cobertura del lote."""
    ranuras = ", ".join(f"{r}={config[r]}" for r in RANURAS if r in config)
    extras = []
    if "temperatura" in config:
        extras.append(f"temp={config['temperatura']}")
    if any(r in config for r in RANURAS):
        extras.append(f"cob={cobertura(config):.0%}")
    extra_txt = (", " + ", ".join(extras)) if extras else ""
    return f"{{{ranuras}{extra_txt}}}"


def contar_configs_validas():
    """Cuántas hojas del espacio pasarían `es_valido`.

    Sirve para ver si `MAX_EVALS` alcanza a cubrir las que el filtro deja
    vivas, o si la búsqueda se corta antes por presupuesto.
    """
    return sum(1 for c in CONFIGS if cobertura(c) < UMBRAL_COBERTURA)


def es_valido(config):
    """Poda si las familias ya acumuladas cubren una fracción grande del lote.

    Distingue poda parcial (faltan ranuras) de poda completa (hoja inválida)
    para ver si el filtro recorta antes de llegar a `evaluar`. La temperatura
    no poda: 0.0 tiene que poder evaluarse.
    """
    if not any(r in config for r in RANURAS):
        return True
    frac = cobertura(config)
    if frac < UMBRAL_COBERTURA:
        return True
    tipo = "completa" if config_completa(config) else "parcial"
    stats[f"podas_{tipo}"] += 1
    print(
        f"  PODA {tipo} (cobertura {frac:.0%} >= {UMBRAL_COBERTURA:.0%}): "
        f"{describir_config(config)}"
    )
    return False


def es_viable(config):
    """Única llamada al oráculo: evalúa la hoja y actualiza la mejor.

    `False` no significa 'inválida': significa 'no mejoró' o 'no hay
    presupuesto'. La validez ya la resolvió `es_valido`.
    """
    global mejor

    if not puede_evaluar():
        stats["cortes_limite"] += 1
        print(
            f"  CORTE límite: no evalúo {describir_config(config)} "
            f"(evaluadas={stats['evaluadas']}/{MAX_EVALS})"
        )
        return False

    print(f"  EVALUAR {describir_config(config)}")
    r = oraculo.evaluar(config, INSTANCIAS, semilla=1)
    stats["evaluadas"] += 1
    historial.append(r.precision)

    mejor_txt = f"{mejor[0]:5.1%}" if mejor else "  n/a"
    if mejor is None or r.precision > mejor[0]:
        mejor = (r.precision, dict(config))
        stats["mejoras"] += 1
        print(
            f"    eval {stats['evaluadas']:3d}/{MAX_EVALS}   esta {r.precision:5.1%}   "
            f"mejor {mejor[0]:5.1%}   ← nueva mejor"
        )
        return True

    stats["sin_mejora"] += 1
    print(
        f"    eval {stats['evaluadas']:3d}/{MAX_EVALS}   esta {r.precision:5.1%}   "
        f"mejor {mejor_txt}   (evaluada pero no mejora)"
    )
    return False


def backtracking(config, profundidad=0):
    """Asigna ranuras en orden. Poda en cuanto `es_valido` falla.

    El orden de `RANURAS` importa: se poda más temprano si las ranuras
    restrictivas van primero. Al volver de la recursión se borra la ranura
    para probar el siguiente índice (el `del` del for).
    """
    indent = "  " * profundidad
    if not puede_evaluar():
        stats["cortes_limite"] += 1
        print(f"{indent}CORTE límite de evaluaciones en {describir_config(config)}")
        return
    if not es_valido(config):
        return

    if config_completa(config):
        print(f"{indent}HOJA válida: {describir_config(config)}")
        es_viable(config)
        return

    siguiente = next(r for r in RANURAS if r not in config)
    print(f"{indent}RAMA {siguiente} desde {describir_config(config)}")
    for i in range(len(CATALOGO[siguiente])):
        config[siguiente] = i
        backtracking(config, profundidad + 1)
        del config[siguiente]
        if not puede_evaluar():
            return

In [ ]:
from oraculo import TEMPERATURAS, espacio

# Tope pedagógico: el oráculo no limita consultas. La partición de búsqueda ya
# viene curada (100 instancias, ver `dividir`), así que se usa entera.
# Con 8 opciones por ranura el ascenso necesita ~70 evaluaciones; 100 deja
# margen para que la poda demuestre algo.
MAX_EVALS = 100
INSTANCIAS = busqueda

# Espacio completo (1024 × 3 temperaturas). El filtro de la celda de arriba
# deja vivas las que no acumulan familias por encima de UMBRAL_COBERTURA.
# 0.0 entra: ya no hay un mínimo de temperatura.
CONFIGS = espacio(TEMPERATURAS)

mejor = None
historial = []
stats = {
    "podas_parcial": 0,
    "podas_completa": 0,
    "evaluadas": 0,
    "mejoras": 0,
    "sin_mejora": 0,
    "cortes_limite": 0,
}

validas_total = contar_configs_validas()

print("=== ANTES DE BUSCAR ===")
print(f"máx. evaluaciones: {MAX_EVALS}")
print(f"instancias/eval:    {len(INSTANCIAS)}")
print(f"espacio:            {len(CONFIGS)} configs  (temperaturas {TEMPERATURAS})")
print(
    f"configs válidas:    {validas_total} / {len(CONFIGS)}  "
    f"(cobertura < {UMBRAL_COBERTURA:.0%}, temperaturas {TEMPERATURAS})"
)
if validas_total > MAX_EVALS:
    print(
        f"⚠ no alcanza el límite para evaluar todas las válidas: "
        f"faltan {validas_total - MAX_EVALS} configs sin probar"
    )
print()

# Un árbol por temperatura. La temp se fija acá y las ranuras se asignan
# adentro: así una temp inválida poda el árbol entero de una vez.
for temp in TEMPERATURAS:
    if not puede_evaluar():
        break
    print(f"--- temperatura {temp} ---")
    backtracking({"temperatura": temp})

print("\n=== RESUMEN ===")
print(f"evaluaciones:       {stats['evaluadas']} / {MAX_EVALS}")
print(f"  mejoras:          {stats['mejoras']}")
print(f"  sin mejora:       {stats['sin_mejora']}")
print(f"podas parciales:    {stats['podas_parcial']}")
print(f"podas completas:    {stats['podas_completa']}")
print(f"cortes por límite:  {stats['cortes_limite']}")
print(f"configs sin evaluar:{max(0, validas_total - stats['evaluadas'])}")

if mejor:
    print("\nmejor configuración:", mejor[1])
else:
    print("\nninguna configuración viable dentro del límite")

### Leer los fallos

Cada resultado trae sus trazas: mírenlas todas las veces que quieran. `violo` es la **primera** restricción que no pasó (el puntaje es todo-o-nada, no hace falta listar las demás). `salida` es el texto que produjo el modelo.

Si quieren ver el prompt **antes** de generar, `ver_prompt(config, instancia)` en `ayudas` arma el texto exacto que recibiría el modelo.


In [ ]:
# Reusa el caché: estas instancias ya se midieron al buscar.
r = oraculo.evaluar(mejor[1], INSTANCIAS, semilla=1)

for t in r.trazas[:3]:
    print("violó:", t["violo"])
    print(t["salida"][:300])
    print("-" * 60)


### Validar la config elegida

Las 300 instancias de validación no se usaron al buscar. Sirve para ver si la config aguanta instancias nuevas, no para elegir otra — si eligen con la validación, dejan de ser un conjunto de prueba.

Validarlas todas tarda; `n` escoge cuántas medir. La muestra es fija: las mismas n en cada llamada. Además imprime qué restricciones se cayeron más.


In [ ]:
# Escojan con cuántas instancias validar: más n = más confiable, pero más lento.
# n=30 es una muestra; n=None (o n=300) mide las 300.
r_val = oraculo.validar(mejor[1], n=30)

print("búsqueda (mejor):", f"{mejor[0]:.1%}")
print("validación:      ", f"{r_val.precision:.1%}")


## 5 · La entrega

Un `entrega.json` con el grupo, la config ganadora y la semana. La nota no sale de este notebook: el profesor corre esa config sobre un test privado. Cambien `G07` y `semana` antes de descargar.


In [ ]:
from ayudas import entrega
from google.colab import files

# grupo: identificador del equipo. semana: número de semana del curso.
entrega(grupo="G07", config=mejor[1], semana=3)
files.download("entrega.json")
